# TripAdvisor Web Scraping

This notebook scrapes TripAdvisor review data for a selected attraction and collects attraction links from a destination listing page.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import os
from bs4 import BeautifulSoup
import numpy as np
import time
import re

## 1. Configure Target Review URL

In [ ]:
init_url = 'https://www.tripadvisor.com/Attraction_Review-g1152744-d10239415-Reviews-High_Spa-Ko_Kut_Trat_Province.html'

## 2. Configure Output File

In [ ]:
file_name = 'High Spa.xlsx'

## 3. Define Review Scraping Function

In [ ]:
def scrape_data(url = init_url):
    
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument('--headless')  # ensure GUI is off
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')

    driver = webdriver.Chrome(options=chrome_options)
    driver.get(url_to_capture)
    
    web_page = driver.page_source
    soup = BeautifulSoup(web_page)
    
    review_tab = soup.find('div', id = 'tab-data-qa-reviews-0')
    
    for remove_elements in review_tab.findChildren('div', class_ = 'hjJJO PJ'):
        remove_elements.decompose()
    
    if len(review_tab.findChildren('span', class_ = 'biGQs _P fiohW fOtGX')) == len(review_tab.findChildren('div', class_ = '_c')):
        review_names = []
        for name_tag in review_tab.findChildren('span', class_ = 'biGQs _P fiohW fOtGX'):
            review_name = name_tag.text
            review_names.append(review_name)
    else:
        index = []
        temps = []
        for tag in review_tab.findChildren('span', class_ = 'biGQs _P fiohW fOtGX'):
            for i, all_message in enumerate(review_tab.findChildren('div', class_ = '_c')):
                if str(tag.text) in str(all_message):
                    if i in index:
                        index.append(i + 1)
                        temps.append((i + 1, tag.text))
                        break
                    else:
                        temps.append((i, tag.text))
                        index.append(i)
                        break

        for i, all_message in enumerate(review_tab.findChildren('div', class_ = '_c')):                
            if i not in index:
                temps.insert(i, (i, np.nan))

        review_names = []
        for index, review_name in temps:
            review_names.append(review_name)
    
    address_list = []
    contributions = []
    for review_data in review_tab.findChildren('div', class_ = '_c'):
        try:
            address = review_data.find('div', class_ = 'biGQs _P pZUbB osNWb').find('span').text
        except:
            address = np.nan
        try:
            contribution = review_data.find('div', class_ = 'biGQs _P pZUbB osNWb').find('span', class_ = 'IugUm').text
        except:
            contribution = np.nan
        try:
            if 'contributions' in address or 'contribution' in address:
                address_list.append(np.nan)
                contributions.append(address)
            else:
                address_list.append(address)
            if str(contribution) ==  'nan':
                pass
            else:
                contributions.append(contribution)
        except:
                address_list.append(np.nan)
                contributions.append(np.nan)
            
    ratings = []
    for rating in review_tab.findChildren('svg', class_ = 'UctUV d H0'):
        rating = rating.attrs['aria-label'].replace(' of 5 bubbles', '').strip()
        ratings.append(rating)
    
    titles = []
    for title in review_tab.findChildren('div', class_ = 'biGQs _P fiohW qWPrE ncFvv fOtGX'):
        titles.append(title.text)
    
    if len(review_tab.findChildren('div', class_ = 'RpeCd')) == len(review_tab.findChildren('div', class_ = '_c')):
        date_deps = []
        travel_types = []
        for date_dep_and_travel_type in review_tab.findChildren('div', class_ = 'RpeCd'):
            try:
                temp = date_dep_and_travel_type.text.split('•')
                if len(temp) == 1:
                    date_dep = temp[0].strip()
                    travel_type = np.nan
                    date_deps.append(date_dep)
                    travel_types.append(travel_type)
                else :
                    date_dep = temp[0].strip()
                    travel_type = temp[1].strip()
                    date_deps.append(date_dep)
                    travel_types.append(travel_type)
            except:
                date_deps.append(np.nan)
                travel_types.append(np.nan)
    
    if len(review_tab.findChildren('div', class_ = 'RpeCd')) == 0:
        date_deps = [np.nan for i in range(0, 10)]
        travel_types = [np.nan for i in range(0, 10)]
    
    if len(review_tab.findChildren('div', class_ = 'RpeCd')) != len(review_tab.findChildren('div', class_ = '_c')):
        index = []
        temps = []
        for tag in review_tab.findChildren('div', class_ = 'RpeCd'):
            for i, all_message in enumerate(review_tab.findChildren('div', class_ = '_c')):
                if str(tag.text) in str(all_message):
                    if i in index:
                        index.append(i + 1)
                        temps.append((i + 1, tag.text))
                        break
                    else:
                        temps.append((i, tag.text))
                        index.append(i)
                        break

        for i, all_message in enumerate(review_tab.findChildren('div', class_ = '_c')):                
            if i not in index:
                temps.insert(i, (i, np.nan))

        date_deps = []
        travel_types = []
        for index, date_dep_and_travel_type in temps:
            try:
                temp = date_dep_and_travel_type.split('•')
                if len(temp) == 1:
                    date_dep = temp[0].strip()
                    travel_type = np.nan
                    date_deps.append(date_dep)
                    travel_types.append(travel_type)
                else:
                    date_dep = temp[0].strip()
                    travel_type = temp[1].strip()
                    date_deps.append(date_dep)
                    travel_types.append(travel_type)
            except:
                date_deps.append(np.nan)
                travel_types.append(np.nan)
        
    text_reviews = []
    for text_review in review_tab.findChildren('div', class_ = 'biGQs _P pZUbB KxBGd'):
        text_reviews.append(text_review.text)
    
    write_dates = []
    for write_date in review_tab.findChildren('div', class_ = 'biGQs _P pZUbB ncFvv osNWb'):
        temp = write_date.text.replace('Written', '').strip()
        write_dates.append(temp)
        
    likes = []
    for like in review_tab.findChildren('span', class_ = 'biGQs _P FwFXZ'):
        likes.append(like.text)
    
    try:
        total_result = review_tab.findChildren('div', class_ = 'Ci')
        total_page = int(re.findall(r'\d+', str(total_result).split('of')[-1].replace(',', ''))[0]) // 10 + 1
    except:
        total_page = 1
        
    time.sleep(1.5)
    
    driver.close()
    
    return review_names, address_list, contributions, ratings, titles, date_deps, travel_types, text_reviews, write_dates, likes, total_page

## 4. Scrape the First Review Page

In [ ]:
review_name_t, address_list_t, contributions_t, ratings_t, titles_t, date_deps_t, travel_types_t, text_reviews_t, write_dates_t, likes_t, total_page_t = scrape_data(url = init_url)

## 5. Generate URLs for Additional Review Pages

In [ ]:
def other_pages(init_url = init_url, total_page = total_page_t):
    
    custom_url = init_url.split('Reviews')
 
    url_generate_other_page = []
    for num in range(2, total_page+1):
        prefix = custom_url[0] + 'Reviews'
        other_page = '-or' + str((num-1) * 10)
        suffix = custom_url[1]
        url_generate_other_page.append(prefix + other_page + suffix)
        
    return url_generate_other_page

## 6. Scrape Remaining Review Pages

In [ ]:
if total_page_t > 1:
    url_other_pages = other_pages(init_url = init_url, total_page = total_page_t)
    for url_page in url_other_pages:
        print(url_page)
        review_names, address_list, contributions, ratings, titles, date_deps, travel_types, text_reviews, write_dates, likes, total_page = scrape_data(url_page)
        print(len(review_names))
        for i in range(0, len(review_names)):
            review_name_t.append(review_names[i])
            address_list_t.append(address_list[i])
            contributions_t.append(contributions[i])
            ratings_t.append(ratings[i])
            titles_t.append(titles[i])
            date_deps_t.append(date_deps[i])
            travel_types_t.append(travel_types[i])
            text_reviews_t.append(text_reviews[i])
            write_dates_t.append(write_dates[i])
            likes_t.append(likes[i])
else:
    pass

## 7. Create Review DataFrame

In [ ]:
# df = pd.DataFrame(zip(review_name_t, address_list_t, contributions_t, ratings_t, titles_t, date_deps_t, travel_types_t, text_reviews_t, write_dates_t, likes_t),
#                  columns = ['review_name', 'address', 'contribution', 'rating', 'title', 'date_dep', 'travel_type', 'text_review', 'write_date', 'like'])

## 8. Preview Scraped Review Data

In [ ]:
df

## 9. Export Review Data to Excel

In [ ]:
df.to_excel(file_name)

## 10. Configure Destination Listing URL

In [ ]:
main_url = 'https://www.tripadvisor.com/Attractions-g580110-Activities-oa0-Ko_Chang_Trat_Province.html'

## 11. Load Destination Listing Page

In [ ]:
driver = webdriver.Edge("msedgedriver.exe")
driver.get(main_url)
web_page = driver.page_source
soup = BeautifulSoup(web_page)

## 12. Extract Attraction Elements

In [ ]:
all_attractions =  soup.find_all('div', class_ = "alPVI eNNhq PgLKC tnGGX")

## 13. Build Full Attraction URLs

In [ ]:
all_links = []
prefix = 'https://www.tripadvisor.com/'
for all_link in all_attractions:
    full_link = prefix + str(all_link.find('a', href = True)['href'])
    all_links.append(full_link)

## 14. Preview Extracted Attraction Links

In [ ]:
all_links